# DataMind: before-fine-tuning baseline on a free Colab T4

Runs the **untuned** `Qwen2.5-Coder-1.5B-Instruct` through the same 50-example Spider evaluation that Claude scored 96% on, and records accuracy, p95 latency and cost per 1,000 queries.

**Run the cells top to bottom, one at a time.** Total time is roughly 15-25 minutes, most of it the one-time `pip install vllm`.

Before you start: **Runtime > Change runtime type > T4 GPU**. A free runtime is temporary (it is deleted after idle time or about 12 hours), so download your results at the end (last cell).

## Step 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

You should see `Tesla T4`, about 15 GB total, and almost nothing used. If it says no GPU, fix the runtime type first.

## Step 2. Upload and unzip the project

Pick **`Datamind_router_colab.zip`** (the one I gave you; it already has the Spider sample and the Claude baseline in it).

In [ ]:
import os
import shutil
import zipfile
from google.colab import files

os.chdir("/content")
uploaded = files.upload()          # choose Datamind_router_colab.zip
zip_name = next(iter(uploaded))

shutil.rmtree("/content/Datamind_router", ignore_errors=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall("/content/Datamind_router")
%cd /content/Datamind_router
!ls
!ls data/spider_sample
!ls src src/eval

You should see `src`, `data`, `tests`, `notebooks`, `results`, and `src/eval` containing `run_eval.py`, `report.py`, `runner.py`. There must be **no** `report.py` or `runner.py` inside `src/datamind`.

## Step 3. Install (about 5-10 minutes)

If Colab shows a **Restart session** button afterwards, click it, then continue from Step 4 (do not repeat this cell).

In [ ]:
!pip install -q vllm
!pip install -q sqlglot pydantic httpx anthropic datasets huggingface-hub
import importlib.metadata as md
print("vllm", md.version("vllm"))

## Step 4. Settings

* `SERVED_NAME` is the name vLLM will answer to. It **must** be the same string you pass as `--local-model` later (the cells below do this for you). Using a different name is what produced the `404 Not Found` you saw.
* `GPU_RATE` is the USD per GPU-hour attributed to the local model. A free Colab T4 costs you nothing, but a $0 local model is not a fair cost story. Look up an on-demand T4 price from a cloud provider's published pricing page, use that number for **every** run, and write down the provider, region and date. Left unset on purpose.

In [ ]:
MODEL       = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SERVED_NAME = "qwen2.5-coder-1.5b-base"
RUN_NAME    = "base_qwen15b_untuned"
GPU_RATE    = None      # <-- set this, e.g. GPU_RATE = 0.35   (USD per GPU-hour; see note above)

## Step 5. Start the vLLM server

This stops any stale server first (a leftover server holding GPU memory is the usual reason a restart fails), starts a fresh one, and waits until it answers. The settings are the ones that worked for you: 4096-token context and `--enforce-eager`.
`--enforce-eager` skips CUDA graphs, so latency is a little higher than it could be. That is fine as long as **every run uses the same setting**, which the notebook does.

In [ ]:
import json
import os
import subprocess
import time
import urllib.request

assert os.path.isdir("/content/Datamind_router/src"), "Run Step 2 first"
LOG = "/content/vllm.log"


def models_endpoint_ok() -> bool:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as r:
            return r.status == 200
    except Exception:
        return False


# Stop any stale vLLM processes (the [v] stops pkill from matching this shell itself).
subprocess.run("pkill -9 -f -i '[v]llm'; sleep 5", shell=True)
subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"])

env = {k: v for k, v in os.environ.items() if k != "VLLM_USE_V1"}   # not used by current vLLM
cmd = [
    "vllm", "serve", MODEL,
    "--served-model-name", SERVED_NAME,
    "--dtype", "half",                     # the T4 has no native bf16
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.80",
    "--enforce-eager",
    "--port", "8000",
]
server = subprocess.Popen(
    cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env, start_new_session=True
)
print("Starting vLLM (the first start downloads the model; allow up to ~10 minutes)...")

deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None or models_endpoint_ok():
        break
    time.sleep(5)

if models_endpoint_ok():
    with urllib.request.urlopen("http://127.0.0.1:8000/v1/models") as r:
        served = [m["id"] for m in json.load(r)["data"]]
    print("READY. Served model names:", served)
    assert SERVED_NAME in served, "SERVED_NAME does not match what vLLM is serving"
else:
    print("NOT READY. Server exit code:", server.poll())
    print("--- error lines (the 'FA version 2' ERROR is a harmless fallback and is filtered out) ---")
    subprocess.run(f"grep -i -E 'error|traceback|memory' {LOG} | grep -v fa_utils | head -25", shell=True)
    print("--- last 25 lines ---")
    subprocess.run(f"tail -n 25 {LOG}", shell=True)

Expected: `READY. Served model names: ['qwen2.5-coder-1.5b-base']`.
In the log, `Cannot use FA version 2 ... Using TRITON_ATTN attention backend` is **normal on a T4** (FlashAttention 2 needs a newer GPU) and is not a failure.
If it says `NOT READY`, send me the printed error lines.

## Step 6. Talk to the server directly (30 seconds)

In [ ]:
import urllib.error


def chat(prompt: str) -> dict:
    body = json.dumps(
        {"model": SERVED_NAME, "messages": [{"role": "user", "content": prompt}],
         "max_tokens": 64, "temperature": 0}
    ).encode()
    req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions", data=body,
        headers={"Content-Type": "application/json"},
    )
    return json.load(urllib.request.urlopen(req, timeout=120))


try:
    r = chat("Write a SQLite query that counts the rows in a table named singer. Reply with SQL only.")
    print(r["choices"][0]["message"]["content"])
    print(r["usage"])
except urllib.error.HTTPError as e:
    print("HTTP", e.code, e.read().decode()[:300])
    print("A 404 here means SERVED_NAME differs from the name vLLM is serving.")

You should see a short SQL query and a `usage` dict. If not, do not go on; fix this first.

## Step 7. Smoke test: 5 examples

Uses the real harness. This also warms the server up so the full run's latency is not skewed by the first request.

In [ ]:
assert GPU_RATE is not None, "Set GPU_RATE in Step 4 first"
!PYTHONPATH=src python -m eval.run_eval --backend local --local-model {SERVED_NAME} --gpu-hourly-usd {GPU_RATE} --limit 5 --run-name base_smoke

Expected: five lines like `[1/5] ok   spider-dev-000 [local]`, then a summary line and `Report written to ...`.
Some `FAIL` lines are fine (the untuned model is weaker). A `FAIL (generation_error)` on **every** line means a server or name problem, so open `results/base_smoke/report.md`.

## Step 8. The full 50-example baseline run (a few minutes)

In [ ]:
assert GPU_RATE is not None, "Set GPU_RATE in Step 4 first"
!PYTHONPATH=src python -m eval.run_eval --backend local --local-model {SERVED_NAME} --gpu-hourly-usd {GPU_RATE} --run-name {RUN_NAME}

## Step 9. Record the numbers

In [ ]:
import datetime
import importlib.metadata as md

run_dir = f"/content/Datamind_router/results/{RUN_NAME}"
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
).stdout.strip()

# Keep the exact serving setup next to the results so the numbers are reproducible.
json.dump(
    {
        "model": MODEL,
        "served_name": SERVED_NAME,
        "vllm_version": md.version("vllm"),
        "vllm_args": cmd[2:],
        "gpu": gpu,
        "gpu_hourly_usd": GPU_RATE,
        "recorded_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    },
    open(f"{run_dir}/serving_config.json", "w"),
    indent=2,
)

s = json.load(open(f"{run_dir}/summary.json"))
lat = s["latency_ms"]
print(f"{RUN_NAME}: accuracy {s['execution_accuracy']:.1%} ({s['n_correct']}/{s['n_examples']})")
print(f"latency ms: p50 {lat['p50']:.0f} | p95 {lat['p95']:.0f} | max {lat['max']:.0f}")
print(f"cost per 1,000 queries: ${s['cost_per_1k_queries_usd']:.2f}   (GPU rate ${GPU_RATE}/h, all queries counted)")
print("status counts:", s["status_counts"])
print("failure categories:", s["failure_counts"])
print("accuracy by complexity:", {k: f"{v['accuracy']:.0%} (n={v['n']})" for k, v in s["accuracy_by_complexity"].items()})

For comparison, the Claude baseline on the same 50 examples: **96% (48/50)**, p50 1,339 ms, **p95 2,487 ms**, **$2.31 per 1,000 queries**.
Do not read much into small gaps: with 50 examples, one query is 2 percentage points.

## Step 10. Download the results (before the runtime disappears)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/baseline_results", "zip", "/content/Datamind_router", "results")
files.download("/content/baseline_results.zip")

Send me `summary.json` (or the whole `results/base_qwen15b_untuned/` folder). Keep `serving_config.json` with it.

### Later, for the fine-tuned model
Change only `MODEL` (your merged model folder), `SERVED_NAME` (for example `qwen2.5-coder-1.5b-spider-lora`) and `RUN_NAME` (for example `tuned_qwen15b`) in Step 4, then run Steps 5-10 again with the **same** `GPU_RATE`.